# Deeploi Example: Iris Classifier

This notebook demonstrates how to use **Deeploi** to package and serve a scikit-learn classifier with a single line of code.

We'll:
1. Train a RandomForest classifier on the Iris dataset
2. Package the model with Deeploi
3. Save the artifact
4. Load and make predictions
5. Serve the model as a FastAPI endpoint

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from deeploi import package, load, deploy

## Step 1: Load Data and Train Model

Load the Iris dataset and split into train/test sets.

In [ ]:
# Load the iris dataset
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Classes: {sorted(y.unique())}")

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Train a RandomForest classifier
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X_train, y_train)

# Evaluate on test set
score = model.score(X_test, y_test)
print(f"Test Accuracy: {score:.3f}")

print(f"\nModel: {model.__class__.__name__}")
print(f"Framework: scikit-learn")

## Step 2: Package the Model with Deeploi

Convert the trained model into a Deeploi package. This automatically:
- Infers the schema from the sample data
- Detects you're using sklearn
- Sets up the classifier for serving

In [ ]:
# Create a Deeploi package from the trained model
pkg = package(model, sample=X_train)

print(f"✓ Created Deeploi package:")
print(f"  {pkg}")
print(f"\n  Framework: {pkg.metadata.framework}")
print(f"  Task: {pkg.metadata.task_type}")
print(f"  Has predict_proba: {pkg.metadata.supports_predict_proba}")
print(f"  Deeploi version: {pkg.metadata.deeploi_version}")
print(f"\n  Features in schema: {len(pkg.schema.features)}")
for feature in pkg.schema.features:
    print(f"    - {feature.name} ({feature.dtype})")

## Step 3: Make Predictions

Test the packaged model. It validates input against the schema and returns predictions with probabilities.

In [ ]:
# Make predictions on test set without probabilities
response = pkg.predict(X_test[:5])
print("Predictions (first 5 samples):")
print(f"  Classes: {response.predictions}")

# Make predictions with probabilities for classifiers
response_proba = pkg.predict(X_test[:5], include_probabilities=True)
print(f"\nPredictions with probabilities:")
print(f"  Predictions: {response_proba.predictions}")
print(f"  Probabilities: {response_proba.probabilities[:2]}")

## Step 4: Save and Reload Artifact

Package can be saved to disk with all dependencies, then reloaded later.

In [ ]:
import os

# Save the package to disk
artifact_path = "artifacts/iris_demo"
pkg.save(artifact_path)
print(f"✓ Saved package to {artifact_path}")

# Show what was saved
if os.path.exists(artifact_path):
    files = os.listdir(artifact_path)
    print(f"\n  Artifact contents:")
    for f in sorted(files):
        file_path = os.path.join(artifact_path, f)
        size = os.path.getsize(file_path)
        print(f"    - {f} ({size:,} bytes)")

In [ ]:
# Load the package back from disk
loaded_pkg = load(artifact_path)
print(f"✓ Loaded package from {artifact_path}")
print(f"  {loaded_pkg}")

# Make predictions with the loaded package
loaded_response = loaded_pkg.predict(X_test[:5], include_probabilities=True)
print(f"\n✓ Made predictions with loaded model:")
print(f"  Predictions: {loaded_response.predictions}")

# Verify they match the original
print(f"\n✓ Predictions match original: {loaded_response.predictions == response_proba.predictions}")

## Step 5: (Optional) Serve the Model as an API

You can serve the model as a FastAPI server with a single line:

```python
pkg.serve(port=8000)
```

Or use the one-liner `deploy()`:

```python
from deeploi import deploy
deploy(model, sample=X_train, port=8000)
```

This creates endpoints:
- **GET /health** — Health check
- **GET /meta** — Model metadata
- **POST /predict** — Make predictions
- **POST /predict_proba** — Get class probabilities

#### Example API call with curl:
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"records": [{"sepal length (cm)": 5.1, "sepal width (cm)": 3.5, "petal length (cm)": 1.4, "petal width (cm)": 0.2}]}'
```

Response:
```json
{
  "predictions": [0],
  "probabilities": [{"0": 0.91, "1": 0.09, "2": 0.0}]
}
```

## Summary

This example showed the complete Deeploi workflow:

✅ **Train** a scikit-learn classifier  
✅ **Package** it with automatic schema inference  
✅ **Predict** with validation and probabilities  
✅ **Save** the artifact to disk  
✅ **Load** and reuse the model  
✅ **Serve** as a FastAPI endpoint (optional)

That's the power of Deeploi: **Turn any sklearn or XGBoost model into a production-ready API in minutes, with zero boilerplate.**

## Bonus: One-Line Deployment

The simplest way to deploy a model is with the `deploy()` function. It combines packaging and serving in a single call:

```python
from deeploi import deploy

deploy(model, sample=X_train, port=8000)
```

That's it! Your model is now running as a FastAPI server.

## Running the Server in Jupyter

Jupyter notebooks have their own event loop, which conflicts with uvicorn's. The best approach is to run the server in a **background thread**, which keeps your notebook responsive while the server handles requests:

In [ ]:
import threading
import time
import webbrowser

def run_server():
    """Run the server in a background thread"""
    import uvicorn
    from deeploi.serving import create_app
    
    app = create_app(pkg)
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="critical")

# Start server in background (non-blocking)
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give server time to start
time.sleep(2)

# Open dashboard in browser
webbrowser.open("http://127.0.0.1:8000")

print("✓ Server started — dashboard opened at http://127.0.0.1:8000")
print("\nServer is running. You can keep using the notebook.")

In [ ]:
import requests
import json

# Test the API
sample_record = {
    "sepal length (cm)": 5.1,
    "sepal width (cm)": 3.5,
    "petal length (cm)": 1.4,
    "petal width (cm)": 0.2,
}

# Make a prediction request
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"records": [sample_record]},
)

print("✓ API Response:")
print(json.dumps(response.json(), indent=2))

# Get metadata
meta_response = requests.get("http://127.0.0.1:8000/meta")
print("\n✓ Model Metadata:")
print(json.dumps(meta_response.json(), indent=2))